# Week 6.4 — Tall Least Squares: LSQR vs. Normal Equations
Constructs a sparse, mildly ill-conditioned tall system and compares LSQR against CG on the normal equations.

In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import lsqr, cg, LinearOperator
import matplotlib.pyplot as plt

## Build the test problem
Constructs an overdetermined ($m=4000 \gg n=1000$), sparse, mildly ill-conditioned least-squares problem $\min_x\|Ax-b\|_2$ with a known sparse true solution `x_true`, so the recovered solutions can be checked against ground truth.

In [2]:
m = 4000
n = 1000
density = 0.01

# sprandn in MATLAB creates a sparse matrix with normally distributed non-zero entries.
A = sp.random(m, n, density=density, format='csr', data_rvs=np.random.randn)

# sprandn(n,1,0.05) creates a sparse vector.
x_true_sparse = sp.random(n, 1, density=0.05, format='csc', data_rvs=np.random.randn)
x_true = x_true_sparse.toarray().flatten()

b = A @ x_true + 1e-3 * np.random.randn(m)
tol = 1e-8
maxit = 2000

## LSQR (matrix-free; uses A and A')
LSQR solves the least-squares problem via Golub-Kahan bidiagonalization, using only products with $A$ and $A^T$ — never forming $A^TA$. This makes it both memory-efficient and numerically more stable than the normal equations, especially when $A$ is ill-conditioned (since it avoids squaring the condition number).

In [3]:
# Note: scipy.sparse.linalg.lsqr does not return a residual history vector.
x_lsqr, istop, itn_l, r1norm = lsqr(A, b, atol=tol, btol=tol, iter_lim=maxit)[:4]

## Normal equations with CG (matrix-free)
For comparison, solves the same problem via the normal equations $A^TAx=A^Tb$ with CG, again applying $A^TA$ matrix-free through a `LinearOperator`. Mathematically equivalent to LSQR's target, but numerically more exposed to ill-conditioning since $\text{cond}(A^TA)=\text{cond}(A)^2$.

In [4]:
AtA_op = LinearOperator((n, n), matvec=lambda x: A.T @ (A @ x), rmatvec=lambda x: A.T @ (A @ x))
rhs = A.T @ b

res_c = []
def callback_cg(xk):
    res_c.append(np.linalg.norm(rhs - AtA_op @ xk))

x0_cg = np.zeros(n)
res_c.append(np.linalg.norm(rhs - AtA_op @ x0_cg))
x_cgne, flag_c = cg(AtA_op, rhs, x0=x0_cg, rtol=tol, maxiter=maxit, callback=callback_cg)

## Compare
Plots the CG-on-normal-equations residual history (LSQR doesn't expose one directly) and reports the relative solution error of both methods against the known true solution.

In [5]:
plt.semilogy(np.arange(len(res_c)), res_c, 'x-', label='CG on A^T A (NE residual)')
plt.xlabel('Iteration')
plt.ylabel('Residual norm')
plt.title('LSQR vs. CG on normal equations')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

print("Note: scipy.sparse.linalg.lsqr does not provide residual history, so it is not plotted.")

print(f'Rel. solution error (LSQR): {np.linalg.norm(x_lsqr - x_true) / np.linalg.norm(x_true):.2e}')
print(f'Rel. solution error (CGNE): {np.linalg.norm(x_cgne - x_true) / np.linalg.norm(x_true):.2e}')

Note: scipy.sparse.linalg.lsqr does not provide residual history, so it is not plotted.
Rel. solution error (LSQR): 8.68e-04
Rel. solution error (CGNE): 8.68e-04


/var/folders/k6/1w07pxzj0mx129drg82_k3_w0000gp/T/ipykernel_73694/3258632687.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
